# Deriving Feynman Rules with Symbolica

This notebook implements the canonical-quantization algorithm for extracting
Feynman vertex factors from a Lagrangian interaction term, using
[Symbolica](https://symbolica.io) as the symbolic engine.

**Algorithm (canonical quantization)**

1. Select an interaction term $\mathcal{L}_{\text{int}} = g_{\alpha_1\dots\alpha_n}\,\partial\dots\partial\,\phi_{\alpha_1}\dots\phi_{\alpha_n}$
2. Attach creation operators $a^\dagger_{\alpha_n}(p_n)\dots a^\dagger_{\alpha_1}(p_1)$
3. Form the vacuum matrix element $\langle 0|\phi_{\alpha_1}(x)\dots\phi_{\alpha_n}(x)\,a^\dagger_{\alpha_n}(p_n)\dots a^\dagger_{\alpha_1}(p_1)|0\rangle$
4. Contract fields with creation operators via canonical (anti)commutation relations
5. Evaluate derivatives on plane waves: $\partial_\mu e^{-ipx} = -ip_\mu\,e^{-ipx}$
6. Remove external wavefunctions and the plane-wave factor
7. Multiply by $i$ to obtain the vertex factor: $V = i\,(\text{remaining coefficient})$

The core logic lives in `code/model_symbolica.py`; this notebook demonstrates
its usage and cross-validates against the spenso-based `code/model.py`.

## Setup

In [1]:
import sys, os
print("Python:", sys.executable)

# Make sure the code/ directory is on the path
sys.path.insert(0, os.path.join(os.getcwd(), "code"))

from model_symbolica import (
    S, Expression, I, pi,
    delta, Delta, pcomp,
    plane_wave, contraction_rule,
    contract_to_full_expression,
    derivative_factors, infer_derivative_targets,
    vertex_factor, simplify_deltas,
)

from IPython.display import Markdown as md, display as dp

def show(expr):
    latex = expr.to_latex().strip().removeprefix("$$").removesuffix("$$")
    dp(md(f"$${latex}$$"))

print("model_symbolica loaded")

Python: /Users/rems/Library/CloudStorage/OneDrive-ETHZurich/ETHz_HS25/MScThesis/.venv/bin/python
model_symbolica loaded


## Step 1 -- Symbols and building blocks

The module defines function symbols:

| Symbol | Meaning |
|--------|---------|
| `phi(alpha, x)` | Scalar field operator |
| `adag(beta, p)` | Creation operator $a^\dagger_\beta(p)$ |
| `U(beta, p)` | External wavefunction (stripped later) |
| `delta(a, b)` | Kronecker delta $\delta_{ab}$ |
| `Delta(q)` | Momentum-conserving Dirac delta |
| `Dot(p, x)` | Minkowski dot product $p \cdot x$ |
| `pcomp(p, mu)` | Momentum component $p_\mu$ |

In [4]:
x = S('x')
p, alpha, beta = S('p', 'alpha', 'beta')

print("Plane wave exp(-i p.x):")
show(plane_wave(p, x))

print("Contraction rule [phi_alpha(x), a^dag_beta(p)]:")
show(contraction_rule(alpha, beta, p, x))

Plane wave exp(-i p.x):


$$$$\exp\!\left(-1𝑖 Dot\!\left(p,x\right)\right)$$$$

Contraction rule [phi_alpha(x), a^dag_beta(p)]:


$$$$\exp\!\left(-1𝑖 Dot\!\left(p,x\right)\right) U\!\left(beta,p\right) delta\!\left(alpha,beta\right)$$$$

## Step 2 -- Wick contractions (permutation sum)

$$
\langle 0|\phi_{\alpha_1}(x)\cdots\phi_{\alpha_n}(x)\,a^\dagger_{\beta_{\sigma(1)}}\cdots|0\rangle
= \sum_\sigma (\pm 1)^{\text{parity}(\sigma)} \prod_i \delta_{\alpha_i \beta_{\sigma(i)}}\, U_{\beta_{\sigma(i)}}(p_{\sigma(i)})\, e^{-i(\sum p) \cdot x}
$$

- Bosons: permanent (all signs $+1$)
- Fermions: determinant ($(-1)^{\text{parity}(\sigma)}$)

## Step 3 -- Derivative momentum factors

Each $\partial_\mu$ on field $k$ gives $(-i)\, p_{k,\mu}$.
The helper `infer_derivative_targets()` builds the needed lists from a
per-field specification.

In [5]:
mu, nu = S('mu', 'nu')
p1, p2 = S('p1', 'p2')

# Two derivatives: d_mu on field 0, d_nu on field 1
deriv_idx, deriv_tgt = infer_derivative_targets([(0, [mu]), (1, [nu])])
print(f"indices = {deriv_idx}, targets = {deriv_tgt}")

df = derivative_factors(deriv_idx, ps_by_field=[p1, p2], derivative_targets=deriv_tgt)
print("Product of momentum factors:")
show(df)

indices = [mu, nu], targets = [0, 1]
Product of momentum factors:


$$$$-pcomp\!\left(p1,mu\right) pcomp\!\left(p2,nu\right)$$$$

## Step 4 -- Full vertex factor pipeline

`vertex_factor(...)` combines all steps:
1. Contract fields with creation operators
2. Multiply by coupling and derivative momentum factors
3. Integrate over $x$: $e^{-i(\sum p)\cdot x} \to (2\pi)^d \Delta(\sum p)$
4. Strip external wavefunctions $U(\beta, p) \to 1$
5. Multiply by $i$

## Examples

In [6]:
x = S('x')
d = S('d')
p1, p2, p3, p4, p5, p6 = S('p1', 'p2', 'p3', 'p4', 'p5', 'p6')
b1, b2, b3, b4, b5, b6 = S('b1', 'b2', 'b3', 'b4', 'b5', 'b6')
mu, nu = S('mu', 'nu')


def show_vertex(title, *, coupling, alphas, betas, ps,
                derivative_indices=(), derivative_targets=None,
                statistics="boson"):
    print("=" * 70)
    print(f"  {title}")
    V = vertex_factor(
        coupling=coupling, alphas=alphas, betas=betas, ps=ps, x=x,
        derivative_indices=derivative_indices,
        derivative_targets=derivative_targets,
        statistics=statistics,
        strip_externals=True, include_delta=True, d=d,
    )
    print(f"  V = {V}")
    show(V)
    return V

### $\phi^4$: expect $V = 24\,i\,\lambda$

In [7]:
phi0 = S('phi0')
lam4 = S('lam4')

V_phi4 = show_vertex(
    "lam4 * phi^4",
    coupling=lam4,
    alphas=[phi0]*4,
    betas=[b1, b2, b3, b4],
    ps=[p1, p2, p3, p4],
)

V_phi4_s = simplify_deltas(V_phi4, {b1: phi0, b2: phi0, b3: phi0, b4: phi0})
print("Simplified:", V_phi4_s)

  lam4 * phi^4
  V = 24𝑖*lam4*(2*𝜋)^d*delta(phi0,b1)*delta(phi0,b2)*delta(phi0,b3)*delta(phi0,b4)*Delta(p1+p2+p3+p4)


$$$$24𝑖 lam4 (2 \pi)^{d} delta\!\left(phi0,b1\right) delta\!\left(phi0,b2\right) delta\!\left(phi0,b3\right) delta\!\left(phi0,b4\right) Delta\!\left(p1+p2+p3+p4\right)$$$$

Simplified: 24𝑖*lam4*(2*𝜋)^d*Delta(p1+p2+p3+p4)


### $\phi^6$: expect $V = 720\,i\,\lambda_6$

In [8]:
lam6 = S('lam6')

V_phi6 = show_vertex(
    "lam6 * phi^6",
    coupling=lam6,
    alphas=[phi0]*6,
    betas=[b1, b2, b3, b4, b5, b6],
    ps=[p1, p2, p3, p4, p5, p6],
)

  lam6 * phi^6
  V = 720𝑖*lam6*(2*𝜋)^d*delta(phi0,b1)*delta(phi0,b2)*delta(phi0,b3)*delta(phi0,b4)*delta(phi0,b5)*delta(phi0,b6)*Delta(p1+p2+p3+p4+p5+p6)


$$$$720𝑖 lam6 (2 \pi)^{d} delta\!\left(phi0,b1\right) delta\!\left(phi0,b2\right) delta\!\left(phi0,b3\right) delta\!\left(phi0,b4\right) delta\!\left(phi0,b5\right) delta\!\left(phi0,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)$$$$

### $\phi^2\chi^2$: expect $V = 4\,i\,g$

In [9]:
chi0 = S('chi0')
g_sym = S('g')

V_phi2chi2 = show_vertex(
    "g * phi^2 chi^2",
    coupling=g_sym,
    alphas=[phi0, phi0, chi0, chi0],
    betas=[b1, b2, b3, b4],
    ps=[p1, p2, p3, p4],
)

V_phi2chi2_s = simplify_deltas(V_phi2chi2, {b1: phi0, b2: phi0, b3: chi0, b4: chi0})
print("Simplified:", V_phi2chi2_s)

  g * phi^2 chi^2
  V = 1𝑖*g*(4*(2*𝜋)^d*delta(phi0,b1)*delta(phi0,b2)*delta(chi0,b3)*delta(chi0,b4)*Delta(p1+p2+p3+p4)+4*(2*𝜋)^d*delta(phi0,b1)*delta(phi0,b3)*delta(chi0,b2)*delta(chi0,b4)*Delta(p1+p2+p3+p4)+4*(2*𝜋)^d*delta(phi0,b1)*delta(phi0,b4)*delta(chi0,b2)*delta(chi0,b3)*Delta(p1+p2+p3+p4)+4*(2*𝜋)^d*delta(phi0,b2)*delta(phi0,b3)*delta(chi0,b1)*delta(chi0,b4)*Delta(p1+p2+p3+p4)+4*(2*𝜋)^d*delta(phi0,b2)*delta(phi0,b4)*delta(chi0,b1)*delta(chi0,b3)*Delta(p1+p2+p3+p4)+4*(2*𝜋)^d*delta(phi0,b3)*delta(phi0,b4)*delta(chi0,b1)*delta(chi0,b2)*Delta(p1+p2+p3+p4))


$$$$1𝑖 g \left(4 (2 \pi)^{d} delta\!\left(phi0,b1\right) delta\!\left(phi0,b2\right) delta\!\left(chi0,b3\right) delta\!\left(chi0,b4\right) Delta\!\left(p1+p2+p3+p4\right)+4 (2 \pi)^{d} delta\!\left(phi0,b1\right) delta\!\left(phi0,b3\right) delta\!\left(chi0,b2\right) delta\!\left(chi0,b4\right) Delta\!\left(p1+p2+p3+p4\right)+4 (2 \pi)^{d} delta\!\left(phi0,b1\right) delta\!\left(phi0,b4\right) delta\!\left(chi0,b2\right) delta\!\left(chi0,b3\right) Delta\!\left(p1+p2+p3+p4\right)+4 (2 \pi)^{d} delta\!\left(phi0,b2\right) delta\!\left(phi0,b3\right) delta\!\left(chi0,b1\right) delta\!\left(chi0,b4\right) Delta\!\left(p1+p2+p3+p4\right)+4 (2 \pi)^{d} delta\!\left(phi0,b2\right) delta\!\left(phi0,b4\right) delta\!\left(chi0,b1\right) delta\!\left(chi0,b3\right) Delta\!\left(p1+p2+p3+p4\right)+4 (2 \pi)^{d} delta\!\left(phi0,b3\right) delta\!\left(phi0,b4\right) delta\!\left(chi0,b1\right) delta\!\left(chi0,b2\right) Delta\!\left(p1+p2+p3+p4\right)\right)$$$$

Simplified: 4𝑖*g*(2*𝜋)^d*Delta(p1+p2+p3+p4)


### Derivative interaction: $g_D\,(\partial_\mu\phi)(\partial_\nu\phi)\,\phi\,\phi$

In [10]:
gD = S('gD')
di, dt = infer_derivative_targets([(0, [mu]), (1, [nu])])

V_deriv = show_vertex(
    "gD * (d_mu phi)(d_nu phi) phi phi",
    coupling=gD,
    alphas=[phi0]*4,
    betas=[b1, b2, b3, b4],
    ps=[p1, p2, p3, p4],
    derivative_indices=di,
    derivative_targets=dt,
)

  gD * (d_mu phi)(d_nu phi) phi phi
  V = -24𝑖*gD*(2*𝜋)^d*delta(phi0,b1)*delta(phi0,b2)*delta(phi0,b3)*delta(phi0,b4)*Delta(p1+p2+p3+p4)*pcomp(p1,mu)*pcomp(p2,nu)


$$$$-24𝑖 gD (2 \pi)^{d} delta\!\left(phi0,b1\right) delta\!\left(phi0,b2\right) delta\!\left(phi0,b3\right) delta\!\left(phi0,b4\right) Delta\!\left(p1+p2+p3+p4\right) pcomp\!\left(p1,mu\right) pcomp\!\left(p2,nu\right)$$$$

### Multi-species: $g_{ijk}\,\phi_i^2\,\phi_j^2\,\phi_k^2$

In [11]:
idx_i, idx_j, idx_k = S('i', 'j', 'k')
gijk = S('gijk')

V_multi = show_vertex(
    "gijk(i,j,k) * phi_i^2 phi_j^2 phi_k^2",
    coupling=gijk(idx_i, idx_j, idx_k),
    alphas=[idx_i, idx_i, idx_j, idx_j, idx_k, idx_k],
    betas=[b1, b2, b3, b4, b5, b6],
    ps=[p1, p2, p3, p4, p5, p6],
)

  gijk(i,j,k) * phi_i^2 phi_j^2 phi_k^2
  V = 1𝑖*(8*(2*𝜋)^d*delta(i,b1)*delta(i,b2)*delta(j,b3)*delta(j,b4)*delta(k,b5)*delta(k,b6)*Delta(p1+p2+p3+p4+p5+p6)+8*(2*𝜋)^d*delta(i,b1)*delta(i,b2)*delta(j,b3)*delta(j,b5)*delta(k,b4)*delta(k,b6)*Delta(p1+p2+p3+p4+p5+p6)+8*(2*𝜋)^d*delta(i,b1)*delta(i,b2)*delta(j,b3)*delta(j,b6)*delta(k,b4)*delta(k,b5)*Delta(p1+p2+p3+p4+p5+p6)+8*(2*𝜋)^d*delta(i,b1)*delta(i,b2)*delta(j,b4)*delta(j,b5)*delta(k,b3)*delta(k,b6)*Delta(p1+p2+p3+p4+p5+p6)+8*(2*𝜋)^d*delta(i,b1)*delta(i,b2)*delta(j,b4)*delta(j,b6)*delta(k,b3)*delta(k,b5)*Delta(p1+p2+p3+p4+p5+p6)+8*(2*𝜋)^d*delta(i,b1)*delta(i,b2)*delta(j,b5)*delta(j,b6)*delta(k,b3)*delta(k,b4)*Delta(p1+p2+p3+p4+p5+p6)+8*(2*𝜋)^d*delta(i,b1)*delta(i,b3)*delta(j,b2)*delta(j,b4)*delta(k,b5)*delta(k,b6)*Delta(p1+p2+p3+p4+p5+p6)+8*(2*𝜋)^d*delta(i,b1)*delta(i,b3)*delta(j,b2)*delta(j,b5)*delta(k,b4)*delta(k,b6)*Delta(p1+p2+p3+p4+p5+p6)+8*(2*𝜋)^d*delta(i,b1)*delta(i,b3)*delta(j,b2)*delta(j,b6)*delta(k,b4)*delta(k,b5)*Delta(p1+p2+

$$$$1𝑖 \left(8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b2\right) delta\!\left(j,b3\right) delta\!\left(j,b4\right) delta\!\left(k,b5\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b2\right) delta\!\left(j,b3\right) delta\!\left(j,b5\right) delta\!\left(k,b4\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b2\right) delta\!\left(j,b3\right) delta\!\left(j,b6\right) delta\!\left(k,b4\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b2\right) delta\!\left(j,b4\right) delta\!\left(j,b5\right) delta\!\left(k,b3\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b2\right) delta\!\left(j,b4\right) delta\!\left(j,b6\right) delta\!\left(k,b3\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b2\right) delta\!\left(j,b5\right) delta\!\left(j,b6\right) delta\!\left(k,b3\right) delta\!\left(k,b4\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b3\right) delta\!\left(j,b2\right) delta\!\left(j,b4\right) delta\!\left(k,b5\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b3\right) delta\!\left(j,b2\right) delta\!\left(j,b5\right) delta\!\left(k,b4\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b3\right) delta\!\left(j,b2\right) delta\!\left(j,b6\right) delta\!\left(k,b4\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b3\right) delta\!\left(j,b4\right) delta\!\left(j,b5\right) delta\!\left(k,b2\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b3\right) delta\!\left(j,b4\right) delta\!\left(j,b6\right) delta\!\left(k,b2\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b3\right) delta\!\left(j,b5\right) delta\!\left(j,b6\right) delta\!\left(k,b2\right) delta\!\left(k,b4\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b4\right) delta\!\left(j,b2\right) delta\!\left(j,b3\right) delta\!\left(k,b5\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b4\right) delta\!\left(j,b2\right) delta\!\left(j,b5\right) delta\!\left(k,b3\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b4\right) delta\!\left(j,b2\right) delta\!\left(j,b6\right) delta\!\left(k,b3\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b4\right) delta\!\left(j,b3\right) delta\!\left(j,b5\right) delta\!\left(k,b2\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b4\right) delta\!\left(j,b3\right) delta\!\left(j,b6\right) delta\!\left(k,b2\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b4\right) delta\!\left(j,b5\right) delta\!\left(j,b6\right) delta\!\left(k,b2\right) delta\!\left(k,b3\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b5\right) delta\!\left(j,b2\right) delta\!\left(j,b3\right) delta\!\left(k,b4\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b5\right) delta\!\left(j,b2\right) delta\!\left(j,b4\right) delta\!\left(k,b3\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b5\right) delta\!\left(j,b2\right) delta\!\left(j,b6\right) delta\!\left(k,b3\right) delta\!\left(k,b4\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b5\right) delta\!\left(j,b3\right) delta\!\left(j,b4\right) delta\!\left(k,b2\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b5\right) delta\!\left(j,b3\right) delta\!\left(j,b6\right) delta\!\left(k,b2\right) delta\!\left(k,b4\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b5\right) delta\!\left(j,b4\right) delta\!\left(j,b6\right) delta\!\left(k,b2\right) delta\!\left(k,b3\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b6\right) delta\!\left(j,b2\right) delta\!\left(j,b3\right) delta\!\left(k,b4\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b6\right) delta\!\left(j,b2\right) delta\!\left(j,b4\right) delta\!\left(k,b3\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b6\right) delta\!\left(j,b2\right) delta\!\left(j,b5\right) delta\!\left(k,b3\right) delta\!\left(k,b4\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b6\right) delta\!\left(j,b3\right) delta\!\left(j,b4\right) delta\!\left(k,b2\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b6\right) delta\!\left(j,b3\right) delta\!\left(j,b5\right) delta\!\left(k,b2\right) delta\!\left(k,b4\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b1\right) delta\!\left(i,b6\right) delta\!\left(j,b4\right) delta\!\left(j,b5\right) delta\!\left(k,b2\right) delta\!\left(k,b3\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b3\right) delta\!\left(j,b1\right) delta\!\left(j,b4\right) delta\!\left(k,b5\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b3\right) delta\!\left(j,b1\right) delta\!\left(j,b5\right) delta\!\left(k,b4\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b3\right) delta\!\left(j,b1\right) delta\!\left(j,b6\right) delta\!\left(k,b4\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b3\right) delta\!\left(j,b4\right) delta\!\left(j,b5\right) delta\!\left(k,b1\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b3\right) delta\!\left(j,b4\right) delta\!\left(j,b6\right) delta\!\left(k,b1\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b3\right) delta\!\left(j,b5\right) delta\!\left(j,b6\right) delta\!\left(k,b1\right) delta\!\left(k,b4\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b4\right) delta\!\left(j,b1\right) delta\!\left(j,b3\right) delta\!\left(k,b5\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b4\right) delta\!\left(j,b1\right) delta\!\left(j,b5\right) delta\!\left(k,b3\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b4\right) delta\!\left(j,b1\right) delta\!\left(j,b6\right) delta\!\left(k,b3\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b4\right) delta\!\left(j,b3\right) delta\!\left(j,b5\right) delta\!\left(k,b1\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b4\right) delta\!\left(j,b3\right) delta\!\left(j,b6\right) delta\!\left(k,b1\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b4\right) delta\!\left(j,b5\right) delta\!\left(j,b6\right) delta\!\left(k,b1\right) delta\!\left(k,b3\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b5\right) delta\!\left(j,b1\right) delta\!\left(j,b3\right) delta\!\left(k,b4\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b5\right) delta\!\left(j,b1\right) delta\!\left(j,b4\right) delta\!\left(k,b3\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b5\right) delta\!\left(j,b1\right) delta\!\left(j,b6\right) delta\!\left(k,b3\right) delta\!\left(k,b4\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b5\right) delta\!\left(j,b3\right) delta\!\left(j,b4\right) delta\!\left(k,b1\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b5\right) delta\!\left(j,b3\right) delta\!\left(j,b6\right) delta\!\left(k,b1\right) delta\!\left(k,b4\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b5\right) delta\!\left(j,b4\right) delta\!\left(j,b6\right) delta\!\left(k,b1\right) delta\!\left(k,b3\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b6\right) delta\!\left(j,b1\right) delta\!\left(j,b3\right) delta\!\left(k,b4\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b6\right) delta\!\left(j,b1\right) delta\!\left(j,b4\right) delta\!\left(k,b3\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b6\right) delta\!\left(j,b1\right) delta\!\left(j,b5\right) delta\!\left(k,b3\right) delta\!\left(k,b4\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b6\right) delta\!\left(j,b3\right) delta\!\left(j,b4\right) delta\!\left(k,b1\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b6\right) delta\!\left(j,b3\right) delta\!\left(j,b5\right) delta\!\left(k,b1\right) delta\!\left(k,b4\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b2\right) delta\!\left(i,b6\right) delta\!\left(j,b4\right) delta\!\left(j,b5\right) delta\!\left(k,b1\right) delta\!\left(k,b3\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b3\right) delta\!\left(i,b4\right) delta\!\left(j,b1\right) delta\!\left(j,b2\right) delta\!\left(k,b5\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b3\right) delta\!\left(i,b4\right) delta\!\left(j,b1\right) delta\!\left(j,b5\right) delta\!\left(k,b2\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b3\right) delta\!\left(i,b4\right) delta\!\left(j,b1\right) delta\!\left(j,b6\right) delta\!\left(k,b2\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b3\right) delta\!\left(i,b4\right) delta\!\left(j,b2\right) delta\!\left(j,b5\right) delta\!\left(k,b1\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b3\right) delta\!\left(i,b4\right) delta\!\left(j,b2\right) delta\!\left(j,b6\right) delta\!\left(k,b1\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b3\right) delta\!\left(i,b4\right) delta\!\left(j,b5\right) delta\!\left(j,b6\right) delta\!\left(k,b1\right) delta\!\left(k,b2\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b3\right) delta\!\left(i,b5\right) delta\!\left(j,b1\right) delta\!\left(j,b2\right) delta\!\left(k,b4\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b3\right) delta\!\left(i,b5\right) delta\!\left(j,b1\right) delta\!\left(j,b4\right) delta\!\left(k,b2\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b3\right) delta\!\left(i,b5\right) delta\!\left(j,b1\right) delta\!\left(j,b6\right) delta\!\left(k,b2\right) delta\!\left(k,b4\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b3\right) delta\!\left(i,b5\right) delta\!\left(j,b2\right) delta\!\left(j,b4\right) delta\!\left(k,b1\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b3\right) delta\!\left(i,b5\right) delta\!\left(j,b2\right) delta\!\left(j,b6\right) delta\!\left(k,b1\right) delta\!\left(k,b4\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b3\right) delta\!\left(i,b5\right) delta\!\left(j,b4\right) delta\!\left(j,b6\right) delta\!\left(k,b1\right) delta\!\left(k,b2\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b3\right) delta\!\left(i,b6\right) delta\!\left(j,b1\right) delta\!\left(j,b2\right) delta\!\left(k,b4\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b3\right) delta\!\left(i,b6\right) delta\!\left(j,b1\right) delta\!\left(j,b4\right) delta\!\left(k,b2\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b3\right) delta\!\left(i,b6\right) delta\!\left(j,b1\right) delta\!\left(j,b5\right) delta\!\left(k,b2\right) delta\!\left(k,b4\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b3\right) delta\!\left(i,b6\right) delta\!\left(j,b2\right) delta\!\left(j,b4\right) delta\!\left(k,b1\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b3\right) delta\!\left(i,b6\right) delta\!\left(j,b2\right) delta\!\left(j,b5\right) delta\!\left(k,b1\right) delta\!\left(k,b4\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b3\right) delta\!\left(i,b6\right) delta\!\left(j,b4\right) delta\!\left(j,b5\right) delta\!\left(k,b1\right) delta\!\left(k,b2\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b4\right) delta\!\left(i,b5\right) delta\!\left(j,b1\right) delta\!\left(j,b2\right) delta\!\left(k,b3\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b4\right) delta\!\left(i,b5\right) delta\!\left(j,b1\right) delta\!\left(j,b3\right) delta\!\left(k,b2\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b4\right) delta\!\left(i,b5\right) delta\!\left(j,b1\right) delta\!\left(j,b6\right) delta\!\left(k,b2\right) delta\!\left(k,b3\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b4\right) delta\!\left(i,b5\right) delta\!\left(j,b2\right) delta\!\left(j,b3\right) delta\!\left(k,b1\right) delta\!\left(k,b6\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b4\right) delta\!\left(i,b5\right) delta\!\left(j,b2\right) delta\!\left(j,b6\right) delta\!\left(k,b1\right) delta\!\left(k,b3\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b4\right) delta\!\left(i,b5\right) delta\!\left(j,b3\right) delta\!\left(j,b6\right) delta\!\left(k,b1\right) delta\!\left(k,b2\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b4\right) delta\!\left(i,b6\right) delta\!\left(j,b1\right) delta\!\left(j,b2\right) delta\!\left(k,b3\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b4\right) delta\!\left(i,b6\right) delta\!\left(j,b1\right) delta\!\left(j,b3\right) delta\!\left(k,b2\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b4\right) delta\!\left(i,b6\right) delta\!\left(j,b1\right) delta\!\left(j,b5\right) delta\!\left(k,b2\right) delta\!\left(k,b3\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b4\right) delta\!\left(i,b6\right) delta\!\left(j,b2\right) delta\!\left(j,b3\right) delta\!\left(k,b1\right) delta\!\left(k,b5\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b4\right) delta\!\left(i,b6\right) delta\!\left(j,b2\right) delta\!\left(j,b5\right) delta\!\left(k,b1\right) delta\!\left(k,b3\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b4\right) delta\!\left(i,b6\right) delta\!\left(j,b3\right) delta\!\left(j,b5\right) delta\!\left(k,b1\right) delta\!\left(k,b2\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b5\right) delta\!\left(i,b6\right) delta\!\left(j,b1\right) delta\!\left(j,b2\right) delta\!\left(k,b3\right) delta\!\left(k,b4\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b5\right) delta\!\left(i,b6\right) delta\!\left(j,b1\right) delta\!\left(j,b3\right) delta\!\left(k,b2\right) delta\!\left(k,b4\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b5\right) delta\!\left(i,b6\right) delta\!\left(j,b1\right) delta\!\left(j,b4\right) delta\!\left(k,b2\right) delta\!\left(k,b3\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b5\right) delta\!\left(i,b6\right) delta\!\left(j,b2\right) delta\!\left(j,b3\right) delta\!\left(k,b1\right) delta\!\left(k,b4\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b5\right) delta\!\left(i,b6\right) delta\!\left(j,b2\right) delta\!\left(j,b4\right) delta\!\left(k,b1\right) delta\!\left(k,b3\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)+8 (2 \pi)^{d} delta\!\left(i,b5\right) delta\!\left(i,b6\right) delta\!\left(j,b3\right) delta\!\left(j,b4\right) delta\!\left(k,b1\right) delta\!\left(k,b2\right) Delta\!\left(p1+p2+p3+p4+p5+p6\right)\right) gijk\!\left(i,j,k\right)$$$$

## Validation against known results

In [12]:
expected_phi4 = 24 * I * lam4 * (2 * pi) ** d * Delta(p1 + p2 + p3 + p4)
assert (
    V_phi4_s.expand().to_canonical_string()
    == expected_phi4.expand().to_canonical_string()
)
print("phi^4: PASS (24 * i * lam4)")

expected_phi2chi2 = 4 * I * g_sym * (2 * pi) ** d * Delta(p1 + p2 + p3 + p4)
assert (
    V_phi2chi2_s.expand().to_canonical_string()
    == expected_phi2chi2.expand().to_canonical_string()
)
print("phi^2 chi^2: PASS (4 * i * g)")

assert "pcomp" in str(V_deriv) and "gD" in str(V_deriv)
print("Derivative: PASS (contains pcomp, gD)")

assert "gijk" in str(V_multi) and "delta" in str(V_multi)
print("Multi-species: PASS (contains gijk, delta)")

print("\nAll validation checks passed.")

phi^4: PASS (24 * i * lam4)
phi^2 chi^2: PASS (4 * i * g)
Derivative: PASS (contains pcomp, gD)
Multi-species: PASS (contains gijk, delta)

All validation checks passed.


## Cross-validation against `model.py` (spenso-based)

The `canonical_vertex()` from `model.py` uses a different representation
(Spenso tensor algebra, OOP metadata classes) but should produce
equivalent numerical factors for the same interactions.

We compare the combinatorial factor (the coefficient that multiplies
$i \times \text{coupling}$) for several benchmark vertices.

In [13]:
from model import (
    Field, Parameter, OperatorFactor, ExternalLeg, LagrangianTerm,
    canonical_vertex, valid_contractions,
    I as I_spenso,
)

# --- phi^4 ---
Phi = Field("phi", kind="scalar", indexed=False, self_conjugate=True)
lampar = Parameter("lam")

L_phi4_spenso = LagrangianTerm(
    "phi4",
    -(lampar.symbol) * Phi() * Phi() * Phi() * Phi(),
    fields=("phi",)*4,
    coefficient=-(lampar.symbol),
    factors=[OperatorFactor(Phi)] * 4,
)

legs_phi4 = [ExternalLeg(Phi, f"p{i+1}") for i in range(4)]

n_contractions_phi4 = len(valid_contractions(L_phi4_spenso, legs_phi4))
v_phi4_spenso = canonical_vertex(L_phi4_spenso, legs_phi4)

print(f"model.py  phi^4: {n_contractions_phi4} contractions, vertex = {v_phi4_spenso}")
print(f"model_symbolica phi^4: 24 permutations, vertex = 24*i*lam4 (after delta simplification)")
assert n_contractions_phi4 == 24
print("  -> Combinatorial factors match: PASS")

# --- phi^2 chi^2 ---
Chi = Field("chi", kind="scalar", indexed=False, self_conjugate=True)
gpar = Parameter("gpar")

L_phi2chi2_spenso = LagrangianTerm(
    "phi2chi2",
    -(gpar.symbol) * Phi() * Phi() * Chi() * Chi(),
    fields=("phi", "phi", "chi", "chi"),
    coefficient=-(gpar.symbol),
    factors=[
        OperatorFactor(Phi), OperatorFactor(Phi),
        OperatorFactor(Chi), OperatorFactor(Chi),
    ],
)

legs_phi2chi2 = [
    ExternalLeg(Phi, "p1"), ExternalLeg(Phi, "p2"),
    ExternalLeg(Chi, "p3"), ExternalLeg(Chi, "p4"),
]

n_contractions_phi2chi2 = len(valid_contractions(L_phi2chi2_spenso, legs_phi2chi2))
v_phi2chi2_spenso = canonical_vertex(L_phi2chi2_spenso, legs_phi2chi2)

print(f"\nmodel.py  phi^2chi^2: {n_contractions_phi2chi2} contractions, vertex = {v_phi2chi2_spenso}")
print(f"model_symbolica phi^2chi^2: 4 valid contractions -> 4*i*g (after delta simplification)")
assert n_contractions_phi2chi2 == 4
print("  -> Combinatorial factors match: PASS")

print("\nCross-validation complete.")

model.py  phi^4: 24 contractions, vertex = -24*I*lam
model_symbolica phi^4: 24 permutations, vertex = 24*i*lam4 (after delta simplification)
  -> Combinatorial factors match: PASS

model.py  phi^2chi^2: 4 contractions, vertex = -4*I*gpar
model_symbolica phi^2chi^2: 4 valid contractions -> 4*i*g (after delta simplification)
  -> Combinatorial factors match: PASS

Cross-validation complete.
